In [ ]:
# ***************************************************************************
#    PySTGEE: Python Implementation for Spatio-Temporal GEE Landslide Modeling
#
#        Original Project     : STGEE (JavaScript/GEE)
#        Original Begin       : 2022-04
#        Original Copyright   : (C) 2022 by Giacomo Titti and Gabriele Nicola Napoli
#        Original Contact     : giacomotitti@gmail.com
#
#        Python Refactoring   : 2026
#        Updated Copyright    : (C) 2026 by Gabriele Nicola Napoli and Giacomo Titti
#        Contacts             : gabrielenicolanapoli@gmail.com, giacomotitti@gmail.com
#
# ***************************************************************************
# ***************************************************************************
#    PySTGEE
#    Copyright (C) 2022 Giacomo Titti, Gabriele Nicola Napoli
#    Copyright (C) 2026 Gabriele Nicola Napoli, Giacomo Titti
#
#    This program is free software: you can redistribute it and/or modify
#    it under the terms of the GNU General Public License as published by
#    the Free Software Foundation, either version 3 of the License, or
#    (at your option) any later version.
#
#    This program is distributed in the hope that it will be useful,
#    but WITHOUT ANY WARRANTY; without even the implied warranty of
#    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
#    GNU General Public License for more details.
#
#    You should have received a copy of the GNU General Public License
#    along with this program.  If not, see <https://www.gnu.org/licenses/>.
# ***************************************************************************

In [ ]:
# @title --- CELL 1: CONFIGURATION ---

# Project settings
EE_PROJECT = 'stgee-dataset'
DATE_COLUMN = 'formatted_'

# Image date range
# IMG_START_DATE = '2002-05-15'
# IMG_END_DATE = '2020-12-31'

IMG_START_DATE = '2011-09-04'
IMG_END_DATE = '2024-09-22'

# Continuous metrics to compute
SELECTED_METRICS_USER = [
    'Elevation', 'Slope', 'Aspect', 'Northness', 'Eastness',
    'PlanCurvature', 'ProfileCurvature', 'Hillshade', 'NDVI', 'NDWI'
]

# Placeholder for categorical metrics
CATEGORICAL_METRICS = ['LULCmajor', 'Litho']

print("Configuration Loaded. Continuous Metrics:", SELECTED_METRICS_USER)

Configuration Loaded. Continuous Metrics: ['Elevation', 'Slope', 'Aspect', 'Northness', 'Eastness', 'PlanCurvature', 'ProfileCurvature', 'Hillshade', 'NDVI', 'NDWI']


In [ ]:
# @title --- CELL 2: SYSTEM SETUP ---
import ee
import geemap
import pandas as pd
import numpy as np
import ipywidgets as widgets
import math
import concurrent.futures
import json
import base64
import sys
import os
from IPython.display import display, Javascript
from google.colab import userdata

# Attempt Service Account Auth, fallback to interactive
try:
    key_content = userdata.get('EE_PRIVATE_KEY')
    service_account_info = json.loads(key_content)
    credentials = ee.ServiceAccountCredentials(service_account_info['client_email'], key_data=key_content)
    ee.Initialize(credentials, project=EE_PROJECT)
    print(f"Connected to Earth Engine via Service Account (Project: {EE_PROJECT}).")
except Exception as e:
    print(f"Service Account Auth Failed: {e}. Falling back to standard auth...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("System Initialization Complete.")

Service Account Auth Failed: Secret EE_PRIVATE_KEY does not exist.. Falling back to standard auth...
System Initialization Complete.


In [ ]:
# @title --- CELL 3: ADVANCED PROCESSING & SMART RECOGNITION ---
import ee
import pandas as pd
import numpy as np
import math
import time
import gc

# Global application state
APP = {
    'training_df': None,
    'prediction_df': None,
    'polygons_fc': None,
    'points_fc': None,
    'model': None,
    'final_predictors': [],
    'original_predictors': [],
    'best_days': None,
    'poly_roles': {},
    'map': None,
    'current_hash': 'default',
    'dummies_map': None,
    'has_categoricals': False # NEW: Tracks if the current run uses CATEGORICAL_METRICS
}

RAW_DATA = {
    'poly_dict': {},
    'points_dict': {},
    'poly_geoms': {},
    'pts_gdf': None,
    'poly_gdfs': {}
}

def check_pre_existing_morphometry(gdf, required_metrics):
    """
    Checks if the input GeoDataFrame already contains the required morphometric
    and spectral columns using fuzzy matching.
    Returns (True, list_of_matching_columns) if a substantial amount is found.
    """
    if gdf is None or gdf.empty:
        return False, []

    found_cols = []
    gdf_cols_lower = [c.lower() for c in gdf.columns]

    # Check for at least some continuous metrics
    matches = 0
    for metric in required_metrics:
        # Check basic metric name (e.g. 'elevation', 'slope', 'ndvi')
        m_lower = metric.lower()
        if any(m_lower in c for c in gdf_cols_lower):
            matches += 1
            # Store the actual original column names that matched
            found_cols.extend([c for c in gdf.columns if m_lower in c.lower() and c not in found_cols])

    # We consider morphometry "pre-existing" if we found at least 4 key continuous metrics
    if matches >= min(4, len(required_metrics)):
        return True, found_cols
    return False, []

def get_missing_categorical_stack(roi, missing_cats):
    """
    Builds a GEE Image containing ONLY the categorical bands we need to extract.
    """
    # Create an empty image to start
    final_image = ee.Image.constant(0).select([])

    # Placeholder logic: In a real scenario, map these to actual GEE assets
    if 'LULCmajor' in missing_cats:
        # Example: Dynamic World or ESRI LULC
        lulc = ee.ImageCollection("ESA/WorldCover/v100").first().select('Map').rename('LULCmajor')
        final_image = final_image.addBands(lulc)
    if 'Litho' in missing_cats:
        # Example: GLiM or local asset
        # You should replace this with your actual Lithology asset path
        litho = ee.Image.constant(0).rename('Litho') # Placeholder
        final_image = final_image.addBands(litho)

    return final_image.toInt()

def calculate_curvatures(elev):
    """Computes Plan and Profile curvature from SRTM elevation."""
    grad = elev.gradient()
    zx, zy = grad.select('x'), grad.select('y')
    grad_zx, grad_zy = zx.gradient(), zy.gradient()
    zxx, zxy = grad_zx.select('x'), grad_zx.select('y')
    zyy = grad_zy.select('y')

    p = zx.pow(2).add(zy.pow(2))
    p_sqrt = p.pow(1.5).max(1e-5)
    q = p.add(1).pow(1.5).max(1e-5)

    plan = zx.pow(2).multiply(zyy).subtract(zx.multiply(zy).multiply(zxy).multiply(2)).add(zy.pow(2).multiply(zxx)).multiply(-1).divide(p_sqrt).rename('PlanCurvature')
    prof = zx.pow(2).multiply(zxx).add(zx.multiply(zy).multiply(zxy).multiply(2)).add(zy.pow(2).multiply(zyy)).divide(p.multiply(q)).rename('ProfileCurvature')
    return plan.addBands(prof).toFloat()

def create_full_stack(roi, include_metrics, include_categoricals):
    """Builds the terrain/spectral index stack based on requested layers."""
    final_image = ee.Image()

    if include_metrics:
        srtm = ee.Image("USGS/SRTMGL1_003").resample('bilinear')
        elev = srtm.toFloat().rename('Elevation')
        slope = ee.Terrain.slope(srtm).toFloat().rename('Slope')
        aspect = ee.Terrain.aspect(srtm).toFloat().rename('Aspect')
        hillshade = ee.Terrain.hillshade(srtm).toFloat().rename('Hillshade')

        aspect_rad = aspect.multiply(math.pi).divide(180)
        north = aspect_rad.cos().toFloat().rename('Northness')
        east = aspect_rad.sin().toFloat().rename('Eastness')

        curvatures = calculate_curvatures(elev)
        end_year = int(IMG_END_DATE.split('-')[0])

        s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")\
            .filterBounds(roi)\
            .filterDate(f'{end_year}-01-01', f'{end_year}-12-31')\
            .select(['B3', 'B4', 'B8'])

        fallback_s2 = ee.Image.constant([0, 0, 0]).rename(['B3', 'B4', 'B8'])
        s2_img = ee.Image(ee.Algorithms.If(s2_col.size().gt(0), s2_col.median(), fallback_s2))

        ndvi = s2_img.normalizedDifference(['B8', 'B4']).toFloat().rename('NDVI')
        ndwi = s2_img.normalizedDifference(['B3', 'B8']).toFloat().rename('NDWI')

        all_bands = elev.addBands([slope, aspect, hillshade, north, east, curvatures, ndvi, ndwi])
        selected_image = all_bands.select(include_metrics)

        # Apply focal mean and standard deviation for textural features
        for name in include_metrics:
            band = selected_image.select(name)
            mean = band.focal_mean(0.5).rename(f'{name}_mean')
            std = band.reduceNeighborhood(ee.Reducer.stdDev(), ee.Kernel.square(1)).rename(f'{name}_stdDev')
            if final_image.bandNames().size().getInfo() == 0:
                final_image = mean.addBands(std)
            else:
                final_image = final_image.addBands([mean, std])

        final_image = final_image.unmask(0.0)

    if include_categoricals:
        cat_stack = get_missing_categorical_stack(roi, include_categoricals)
        if final_image.bandNames().size().getInfo() == 0:
            final_image = cat_stack
        else:
            final_image = final_image.addBands(cat_stack)

    return final_image

def extract_covariates_fast(points_df, log_widget, missing_metrics, missing_cats):
    """Extracts raster values at point locations for the explicitly requested bands."""
    def log(msg): log_widget.write(msg)

    features = []
    for _, row in points_df.iterrows():
        feat = ee.Feature(ee.Geometry.Point([row['lon'], row['lat']]), {
            'poly_uid': str(row['poly_uid']),
            'P/A': int(row['P/A']),
            'event_date': str(row['event_date'])
        })
        features.append(feat)

    fc = ee.FeatureCollection(features)
    roi = fc.geometry()

    stack = create_full_stack(roi, missing_metrics, missing_cats)

    for attempt in range(3):
        try:
            # We use first() for categoricals to get the exact class, and mean/std were pre-calculated above
            sampled = stack.reduceRegions(
                collection=fc,
                reducer=ee.Reducer.first(),
                scale=30,
                tileScale=16
            ).getInfo()

            if 'features' in sampled:
                rows = [f['properties'] for f in sampled['features']]
                if rows:
                    df_res = pd.DataFrame(rows)
                    base = ['P/A', 'event_date', 'poly_uid']
                    metrics = [c for c in df_res.columns if c not in base and '.geo' not in c and 'system:index' not in c]
                    metrics.sort()
                    for b in base:
                        if b not in df_res.columns: df_res[b] = 0
                    return df_res[base + metrics].fillna(0.0)
            return pd.DataFrame()
        except Exception as e:
            if attempt < 2:
                log(f"     [API Error] Retrying {attempt+2}/3...")
                time.sleep(3)
            else:
                log(f"     [GEE API Failed]: {e}")
                return pd.DataFrame()

In [ ]:
# @title --- CELL 4: RAINFALL EXTRACTION (GSMaP) ---
def download_rainfall_data(log_widget):
    """Extracts historical 7-day and 14-day rainfall accumulation using GSMaP."""
    import concurrent.futures
    import pandas as pd
    import ee
    import time
    import gc

    df = APP['training_df']
    if df is None:
        with log_widget: print("ERROR: Calculate static morphometry first.")
        return pd.DataFrame()

    with log_widget: print("--- STARTING HISTORICAL RAINFALL EXTRACTION (GSMaP) ---")

    unique_dates = df['event_date'].unique()
    unique_polys = df[['poly_uid']].drop_duplicates()

    features = []
    for _, r in unique_polys.iterrows():
        try:
            lon = float(str(r['poly_uid']).split('_')[0]) / 1e7
            lat = float(str(r['poly_uid']).split('_')[1]) / 1e7
            features.append(ee.Feature(ee.Geometry.Point([lon, lat]), {'poly_uid': str(r['poly_uid'])}))
        except: pass

    CHUNK_SIZE = 2000
    ee_chunks = [ee.FeatureCollection(features[i : i + CHUNK_SIZE]) for i in range(0, len(features), CHUNK_SIZE)]

    gsmap = ee.ImageCollection("JAXA/GPM_L3/GSMaP/v8/operational").select('hourlyPrecipRateGC')

    def extract_for_date(date_str):
        d_target = ee.Date(date_str)
        rn7_col = gsmap.filterDate(d_target.advance(-7, 'day'), d_target)
        rn14_col = gsmap.filterDate(d_target.advance(-14, 'day'), d_target)

        rn7 = ee.Image(ee.Algorithms.If(rn7_col.size().gt(0), rn7_col.sum(), ee.Image(0))).rename('Rn7_m')
        rn14 = ee.Image(ee.Algorithms.If(rn14_col.size().gt(0), rn14_col.sum(), ee.Image(0))).rename('Rn14_m')
        combined_img = rn7.addBands(rn14)

        date_results = []
        for chunk in ee_chunks:
            for attempt in range(3):
                try:
                    res = combined_img.reduceRegions(collection=chunk, reducer=ee.Reducer.first(), scale=5000, tileScale=4).getInfo()
                    for f in res.get('features', []):
                        props = f.get('properties', {})
                        date_results.append({'poly_uid': props.get('poly_uid'), 'proc_date': date_str, 'Rn7_m': props.get('Rn7_m', 0), 'Rn14_m': props.get('Rn14_m', 0)})
                    break
                except Exception as e:
                    time.sleep(2)
        return date_results

    all_rain_data = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
        future_to_date = {executor.submit(extract_for_date, d_str): d_str for d_str in unique_dates}
        for count, future in enumerate(concurrent.futures.as_completed(future_to_date), 1):
            try:
                res = future.result()
                if res: all_rain_data.extend(res)
            except Exception as e: pass

    if not all_rain_data: return df
    rain_df = pd.DataFrame(all_rain_data)

    with log_widget: print("    > Merging rainfall data into memory...")
    # Polars implementation for optimized memory handling
    import polars as pl
    pl_df = pl.from_pandas(df)
    pl_rain = pl.from_pandas(rain_df)

    final_pl = pl_df.join(pl_rain, left_on=['poly_uid', 'event_date'], right_on=['poly_uid', 'proc_date'], how='left')
    final = final_pl.to_pandas()

    del pl_df, pl_rain, rain_df
    gc.collect()

    final = final.fillna({'Rn7_m': 0, 'Rn14_m': 0})
    with log_widget: print("--- HISTORICAL RAINFALL MERGED SUCCESSFULLY ---")
    return final

In [ ]:
# @title --- CELL 5: SPATIO-TEMPORAL PREDICTION (PYTHON) ---
# -----------------------------------------------------------------------------
# This cell implements the STGEE spatio-temporal prediction logic in Python.
# It translates the original JavaScript approach (rainCalculator.js) into Python.
# ECMWF is used for future dates; JAXA GSMaP is used for historical dates.
# BUGFIX: Handles GEE single-band reducer renaming ('mean' instead of 'Rn7_m').
# -----------------------------------------------------------------------------

import pandas as pd
import numpy as np
import datetime
import time
import os
import gc
import ee

def extract_coordinates(uid):
    """Decode poly_uid to (longitude, latitude)."""
    uid_str = str(uid)
    if '_' in uid_str:
        parts = uid_str.split('_')
        try:
            return float(parts[0]), float(parts[1])
        except ValueError:
            try:
                return float(parts[0]) / 1e7, float(parts[1]) / 1e7
            except:
                return 0.0, 0.0
    else:
        try:
            return int(uid_str) / 1e7, 0.0
        except:
            return 0.0, 0.0

def get_rainfall_image(target_date_str, days, source='JAXA'):
    """Builds the EE Image for cumulative rainfall over the specified days."""
    d_target = ee.Date(target_date_str)

    # Forecast data (Future)
    if source == 'ECMWF':
        dataset = ee.ImageCollection("ECMWF/NRT_FORECAST/IFS/OPER").select("total_precipitation_rate_sfc")
        start = d_target.advance(-days, 'day')
        end = d_target.advance(1, 'day')
        col = dataset.filterDate(start, end)
        # Convert mm/s -> mm per hour (3600 seconds)
        precip_mm_h = col.map(lambda img: img.multiply(3600).rename('precip').copyProperties(img, img.propertyNames()))
        img = ee.Image(ee.Algorithms.If(precip_mm_h.size().gt(0), precip_mm_h.sum(), ee.Image(0)))
        return img.unmask(0).rename(f'Rn{days}_m').toFloat()
    else:
        # Historical data (Past)
        dataset = ee.ImageCollection("JAXA/GPM_L3/GSMaP/v8/operational").select('hourlyPrecipRateGC')
        start = d_target.advance(-days, 'day')
        end = d_target
        col = dataset.filterDate(start, end)

        img = ee.Image(ee.Algorithms.If(col.size().gt(0), col.sum(), ee.Image(0)))
        return img.unmask(0).rename(f'Rn{days}_m').toFloat()

def extract_rainfall_for_polygons(polygons_df, target_date_str, days, source='JAXA', log_widget=None, chunk_size=1000):
    """
    Extracts cumulative rainfall for each polygon centroid using GEE reduceRegions.
    Guarantees no KeyErrors by pre-allocating a 0.0 dictionary for all UIDs.
    """
    def _log(msg):
        if log_widget:
            with log_widget: print(msg)
        else:
            print(msg)

    rain_col = f'Rn{days}_m'
    _log(f"    [LOG] Building {source} image for {days} days ending on {target_date_str}...")
    rain_img = get_rainfall_image(target_date_str, days, source=source)

    df_coords = polygons_df[['poly_uid']].drop_duplicates().copy()
    df_coords[['lon', 'lat']] = df_coords['poly_uid'].apply(lambda x: pd.Series(extract_coordinates(x)))

    # 1. PRE-ALLOCATE dictionary with 0.0 for every single poly_uid
    rain_dict = {}
    features_data = []
    for _, row in df_coords.iterrows():
        uid_str = str(row['poly_uid'])
        rain_dict[uid_str] = 0.0
        features_data.append({
            'uid': uid_str,
            'lon': float(row['lon']),
            'lat': float(row['lat'])
        })

    total = len(features_data)
    _log(f"    [LOG] Extracting rainfall for {total} points in chunks of {chunk_size}...")

    # 2. Extract from Earth Engine
    for i in range(0, total, chunk_size):
        chunk_data = features_data[i:i+chunk_size]
        ee_features = [ee.Feature(ee.Geometry.Point([d['lon'], d['lat']]), {'poly_uid': d['uid']}) for d in chunk_data]
        fc_chunk = ee.FeatureCollection(ee_features)

        for attempt in range(3):
            try:
                result = rain_img.reduceRegions(
                    collection=fc_chunk,
                    reducer=ee.Reducer.mean(),
                    scale=2000,
                    tileScale=4
                ).getInfo()

                features = result.get('features', [])
                if len(features) > 0:
                    # Stampa le chiavi del primo elemento per debug
                    if i == 0 and attempt == 0:
                        props_keys = list(features[0].get('properties', {}).keys())
                        _log(f"    [DEBUG] Earth Engine returned properties: {props_keys}")

                    for f in features:
                        props = f.get('properties', {})
                        uid = props.get('poly_uid')

                        # FIX: GEE renames the output to 'mean' when reducing a single band!
                        val = props.get(rain_col)
                        if val is None:
                            val = props.get('mean')
                        if val is None:
                            val = props.get('first')

                        if uid is not None and val is not None:
                            rain_dict[str(uid)] = float(val)
                break
            except Exception as e:
                if attempt == 2:
                    _log(f"    [!] Chunk {i//chunk_size + 1} failed: {e}. Kept as 0.0.")
                else:
                    time.sleep(2 ** attempt)

    # 3. Safely convert to DataFrame and merge
    rain_df = pd.DataFrame(list(rain_dict.items()), columns=['poly_uid', rain_col])

    polygons_df = polygons_df.copy()
    polygons_df['poly_uid'] = polygons_df['poly_uid'].astype(str)
    rain_df['poly_uid'] = rain_df['poly_uid'].astype(str)

    # Drop pre-existing rain column to avoid _x and _y suffixing during merge
    if rain_col in polygons_df.columns:
        polygons_df = polygons_df.drop(columns=[rain_col])

    merged = polygons_df.merge(rain_df, on='poly_uid', how='left')
    merged[rain_col] = merged[rain_col].fillna(0.0)

    _log(f"    [LOG] Extraction complete. Min Rain: {merged[rain_col].min():.4f}, Max Rain: {merged[rain_col].max():.4f}")
    return merged

def predict_spacetime(target_date_str, static_polygons_df, model, original_predictors,
                      dummies_map, best_days, train_ref_rain=None, log_widget=None):
    """
    Performs the complete spatio-temporal prediction as defined in rainCalculator.js
    """
    def _log(msg):
        if log_widget:
            with log_widget: print(msg)
        else:
            print(msg)

    if model is None:
        raise ValueError("Model is None. Run Calibration first.")
    if static_polygons_df is None or static_polygons_df.empty:
        raise ValueError("Prediction DataFrame is empty. Run 'Calc Morphometry'.")

    _log(f"--- SPATIO-TEMPORAL PREDICTION FOR {target_date_str} (window {best_days} days) ---")

    df = static_polygons_df.copy()

    # 1. Base Model Prediction (Static)
    # FIX: uses CATEGORICAL_METRICS to identify categorical columns
    cat_cols = [c for c in CATEGORICAL_METRICS if c in original_predictors]
    X_static, _, _ = encode_categoricals(
        df[original_predictors],
        original_predictors,
        cat_cols,
        dummies_map=dummies_map
    )
    X_static = X_static.fillna(0)
    probs = model.predict_proba(X_static)[:, 1]
    df['Susceptibility_Prob'] = probs

    # 2. Dynamic Rainfall Extraction
    target_dt = datetime.datetime.strptime(target_date_str, '%Y-%m-%d').date()
    is_future = target_dt >= datetime.date.today()
    source = 'ECMWF' if is_future else 'JAXA'

    df_with_rain = extract_rainfall_for_polygons(
        df, target_date_str, best_days, source=source, log_widget=log_widget, chunk_size=1000
    )

    # 3. Dynamic Susceptibility Calculation
    rain_col = f'Rn{best_days}_m'

    if train_ref_rain is None or train_ref_rain <= 0:
        train_ref_rain = 200.0

    df_with_rain['Final_Dynamic_Susceptibility'] = 1.0 - (1.0 - df_with_rain['Susceptibility_Prob']) * np.exp(-df_with_rain[rain_col] / train_ref_rain)

    df_with_rain.rename(columns={rain_col: 'Rn_m'}, inplace=True)

    return df_with_rain[['poly_uid', 'Susceptibility_Prob', 'Rn_m', 'Final_Dynamic_Susceptibility']]

print("CELL 5: Spatio-temporal prediction logic loaded successfully.")

CELL 5: Spatio-temporal prediction logic loaded successfully.


In [ ]:
!pip install fiona rasterio polars joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 55.0 MB/s eta 0:00:00


In [ ]:
# @title --- CELL 6: MAIN APPLICATION & INTERACTIVE UI ---

# -----------------------------------------------------------------------------
# IMPORTS
# -----------------------------------------------------------------------------
import fiona
import rasterio
import polars as pl
import joblib
import ee, geemap, geemap.coreutils
import ipywidgets as widgets
from ipyleaflet import WidgetControl, ZoomControl, FullScreenControl, ImageOverlay, GeoData
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix, auc, f1_score, cohen_kappa_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from IPython.display import display, Javascript, clear_output
import pandas as pd
import geopandas as gpd
import numpy as np
import os, glob, shutil, time, gc, warnings, json
import datetime
from shapely.geometry import shape, mapping, Point
from shapely.ops import transform as shapely_transform
import shapely.ops
import pyproj
from functools import partial
import matplotlib.colors as mcolors
import rasterio.features
import rasterio.transform
from PIL import Image
import base64
from io import BytesIO
from scipy.spatial import KDTree

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

geemap.coreutils.get_env_var = lambda key: None
import sys
sys.setrecursionlimit(3000)

# -----------------------------------------------------------------------------
# GLOBAL CONSTANTS & APP STATE
# -----------------------------------------------------------------------------
VIS_PALETTE = ['#006b0b', '#1b7b25', '#4e9956', '#dbeadd', '#ffffff', '#f0b2ae', '#eb958f', '#df564d', '#d10e00']
PALETTE_CONFUSION = ['#D10E00', '#DF564D', '#DBEADD', '#006B0B']

INPUT_CACHE_DIR = '/content/drive/MyDrive/Morfometry_Results/Input_Cache'
RESULTS_DIR = '/content/drive/MyDrive/Morfometry_Results'
RASTER_DIR = os.path.join(RESULTS_DIR, 'Rasters')

APP = {
    'training_df': None,
    'prediction_df': None,
    'panels': {'calib': None, 'valid': None, 'pred': None, 'legend': None},
    'model': None,
    'final_predictors': [],
    'original_predictors': [],
    'best_days': None,
    'poly_roles': {},
    'map': None,
    'current_hash': 'default',
    'dummies_map': None
}

RAW_DATA = {
    'poly_dict': {},
    'points_dict': {},
    'poly_geoms': {},
    'pts_gdf': None,
    'poly_gdfs': {}
}

# -----------------------------------------------------------------------------
# GEOMETRY UTILITIES
# -----------------------------------------------------------------------------
def repair_geometry(geom):
    """Fix invalid geometries (buffer(0), convert collections, etc.)."""
    try:
        if geom is None or geom.is_empty:
            return None
        if not geom.is_valid:
            geom = geom.buffer(0)
        if geom.geom_type == 'MultiPolygon':
            geom = geom.buffer(0)
        if geom.geom_type == 'GeometryCollection':
            polygons = [g for g in geom.geoms if g.geom_type in ['Polygon', 'MultiPolygon']]
            if polygons:
                geom = polygons[0] if len(polygons) == 1 else shapely.ops.unary_union(polygons)
            else:
                return None
        if geom.is_valid and not geom.is_empty and geom.geom_type in ['Polygon', 'MultiPolygon']:
            return geom
        else:
            return None
    except:
        return None

def extract_coordinates(uid):
    """Decode poly_uid to (lon, lat)."""
    uid_str = str(uid)
    if '_' in uid_str:
        parts = uid_str.split('_')
        try:
            return float(parts[0]), float(parts[1])
        except ValueError:
            try:
                return float(parts[0]) / 1e7, float(parts[1]) / 1e7
            except:
                return 0.0, 0.0
    else:
        try:
            return int(uid_str) / 1e7, 0.0
        except:
            return 0.0, 0.0

# -----------------------------------------------------------------------------
# CHECKPOINT & CACHE MANAGEMENT
# -----------------------------------------------------------------------------
def load_checkpoint_state(phase, train_hash, log_widget):
    """Restore model or data from Drive cache."""
    model_cache_path = os.path.join(RESULTS_DIR, f"MASTER_MODEL_{train_hash}.joblib")
    master_train_csv = os.path.join(RESULTS_DIR, 'Cache', f"MASTER_TRAIN_{train_hash}.csv")

    if phase in ['valid', 'pred']:
        if APP.get('model') is None:
            if os.path.exists(model_cache_path):
                with log_widget: print(f"    Restoring model from checkpoint '{train_hash}'...")
                cached_data = joblib.load(model_cache_path)
                APP['model'] = cached_data['model']
                APP['final_predictors'] = cached_data['predictors']
                APP['original_predictors'] = cached_data.get('original_predictors', [])
                APP['best_days'] = cached_data.get('best_days', 7)
                APP['dummies_map'] = cached_data.get('dummies_map', None)
            else:
                raise FileNotFoundError("Model not found on Drive. Please run Calibration first.")

    if phase == 'valid' and APP.get('training_df') is None:
        if os.path.exists(master_train_csv):
            with log_widget: print("    Restoring training dataset from checkpoint...")
            APP['training_df'] = pd.read_csv(master_train_csv, low_memory=False)
        else:
            raise FileNotFoundError("Training dataset not found. Please run 'Calc Morphometry'.")

    if phase == 'pred' and APP.get('prediction_df') is None:
        cache_dir = '/content/drive/MyDrive/Morfometry_Results/Cache'
        pred_files = glob.glob(os.path.join(cache_dir, "*_PRED_static.csv"))
        if pred_files:
            with log_widget: print("    Restoring static morphology dataset from checkpoint...")
            pred_list = [pd.read_csv(f, low_memory=False) for f in pred_files]
            APP['prediction_df'] = pd.concat(pred_list, ignore_index=True)
        else:
            raise FileNotFoundError("Prediction morphology not found. Please run 'Calc Morphometry'.")

    return True


def keep_colab_alive():
    """Prevent Colab from disconnecting due to inactivity."""
    display(Javascript('''
        function ClickConnect() {
            var connect_btn = document.querySelector("colab-connect-button");
            if (connect_btn && connect_btn.shadowRoot) {
                var actual_btn = connect_btn.shadowRoot.querySelector("#connect");
                if (actual_btn) {
                    actual_btn.click();
                } else {
                    var toolbar_btn = connect_btn.shadowRoot.querySelector("colab-toolbar-button");
                    if (toolbar_btn) toolbar_btn.click();
                }
            }
        }
        setInterval(ClickConnect, 60000);
    '''))

def ensure_drive_mounted(log_widget=None):
    """Mount Google Drive if not already mounted."""
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            if log_widget:
                with log_widget: print("Mounting Google Drive. Please authorize...")
            drive.mount('/content/drive')
    except ImportError:
        pass
    except Exception as e:
        if log_widget:
            with log_widget: print(f"Drive mount error: {e}")
    os.makedirs(RESULTS_DIR, exist_ok=True)
    os.makedirs(RASTER_DIR, exist_ok=True)
    os.makedirs(INPUT_CACHE_DIR, exist_ok=True)


# -----------------------------------------------------------------------------
# METRICS, ENCODING, RASTERIZATION HELPERS
# -----------------------------------------------------------------------------
def calc_confusion_class(row, pred_col, true_col='P/A'):
    p, t = int(row[pred_col]), int(row[true_col])
    if p == 1 and t == 0: return 0
    if p == 0 and t == 0: return 1
    if p == 0 and t == 1: return 2
    if p == 1 and t == 1: return 3
    return 1

def write_to_drive_direct(df_chunk, path, is_first):
    mode = 'w' if is_first else 'a'
    with open(path, mode) as f:
        df_chunk.to_csv(f, index=False, header=is_first)
        f.flush()
        os.fsync(f.fileno())

def create_save_to_drive_button(source_file_path, file_prefix, title="Save GeoJSON to Drive"):
    """Create a button that copies a file to the results folder on Drive."""
    btn = widgets.Button(
        description=title,
        layout=widgets.Layout(width='auto', padding='0px 15px', height='35px'),
        style={'button_color': '#28a745', 'font_weight': 'bold', 'text_color': 'white'}
    )
    def on_click(b):
        keep_colab_alive()
        ensure_drive_mounted(out_log)
        with out_log:
            btn.description = "Saving..."
            btn.disabled = True
            try:
                ext = os.path.splitext(source_file_path)[1]
                dest_filename = f"{file_prefix}{ext}"
                dest_path = os.path.join(RESULTS_DIR, dest_filename)
                shutil.copy(source_file_path, dest_path)
                print(f"Saved to Drive: {dest_path}")
                btn.description = "Saved to Drive"
                btn.style.button_color = '#1b7b25'
            except Exception as e:
                print(f"Export Error: {e}")
                btn.description = "Error!"
                btn.style.button_color = '#dc3545'
            finally:
                btn.disabled = False
    btn.on_click(on_click)
    return btn

def calculate_advanced_metrics_polars(y_true, y_probs):
    """Compute AUC, best threshold, F1, Kappa, Accuracy using Polars."""
    df = pl.DataFrame({'y': y_true, 'prob': y_probs})
    df = df.sort('prob', descending=True)
    total_p = df['y'].sum()
    total_n = df.height - total_p
    roc_df = df.with_columns([
        pl.col('y').cum_sum().alias('tp'),
        (1 - pl.col('y')).cum_sum().alias('fp')
    ]).with_columns([
        (pl.col('tp') / total_p).alias('tpr'),
        (pl.col('fp') / total_n).alias('fpr')
    ])
    fpr = roc_df['fpr'].to_numpy()
    tpr = roc_df['tpr'].to_numpy()
    roc_thresh = roc_df['prob'].to_numpy()
    roc_auc = auc(fpr, tpr)
    best_idx = np.argmax(tpr - fpr)
    best_thresh = roc_thresh[best_idx]
    y_pred_opt = (y_probs >= best_thresh).astype(int)
    return {
        'auc': roc_auc, 'best_thresh': best_thresh,
        'f1': f1_score(y_true, y_pred_opt, zero_division=0),
        'kappa': cohen_kappa_score(y_true, y_pred_opt),
        'acc': accuracy_score(y_true, y_pred_opt),
        'fpr': fpr, 'tpr': tpr, 'y_pred_opt': y_pred_opt
    }

def encode_categoricals(df, predictor_cols, cat_cols, dummies_map=None):
    """One‑hot encode categorical variables, preserving a mapping for later reuse."""
    df = df.copy()
    if not cat_cols:
        return df[predictor_cols], predictor_cols, None

    if dummies_map is not None:
        for col, cats in dummies_map.items():
            if col not in df.columns:
                df[col] = 0
            for cat in cats:
                dummy_col = f"{col}_{cat}"
                df[dummy_col] = (df[col] == cat).astype(int)
            df.drop(columns=[col], inplace=True, errors='ignore')
        all_dummies = [f"{col}_{cat}" for col, cats in dummies_map.items() for cat in cats]
        for c in all_dummies:
            if c not in df.columns:
                df[c] = 0
        new_preds = [c for c in predictor_cols if c not in cat_cols] + all_dummies
        return df[new_preds], new_preds, dummies_map
    else:
        new_map = {}
        new_preds = list(predictor_cols)
        for col in cat_cols:
            if col in df.columns:
                unique_vals = sorted(df[col].dropna().unique())
                new_map[col] = list(unique_vals)
                for val in unique_vals:
                    dummy_col = f"{col}_{val}"
                    df[dummy_col] = (df[col] == val).astype(int)
                df.drop(columns=[col], inplace=True)
                new_preds.remove(col)
                new_preds.extend([f"{col}_{v}" for v in unique_vals])
        final_cols = [c for c in new_preds if c in df.columns]
        return df[final_cols], final_cols, new_map


# -----------------------------------------------------------------------------
# MAP RENDERING (rasterize polygons to ImageOverlay)
# -----------------------------------------------------------------------------
def add_map_layer_geemap_native(df_results, val_col, layer_name, palette, is_categorical=False, dynamic_scale=False):
    """Render a categorical or continuous variable as a raster overlay on the map."""
    try:
        if 'poly_uid' not in df_results.columns:
            return

        train_hash = APP.get('current_hash', 'default')
        safe_name = f"{layer_name.replace(' ', '_').replace(':', '')}_{train_hash}"
        png_path = os.path.join(RASTER_DIR, f"{safe_name}.png")
        bounds_path = os.path.join(RASTER_DIR, f"{safe_name}_bounds.json")

        # Remove existing layer with same name
        for l in APP['map'].layers:
            if layer_name in getattr(l, 'name', ''):
                APP['map'].remove_layer(l)

        # Load cached raster if exists
        if os.path.exists(png_path) and os.path.exists(bounds_path):
            with out_log: print(f"    Loading cached raster: '{layer_name}'")
            with open(png_path, "rb") as img_file:
                img_str = base64.b64encode(img_file.read()).decode()
            with open(bounds_path, "r") as b_file:
                bounds = json.load(b_file)
            image_url = f"data:image/png;base64,{img_str}"
            image_layer = ImageOverlay(url=image_url, bounds=bounds, name=layer_name, opacity=0.85)
            APP['map'].add_layer(image_layer)
            return

        # Prepare geometries and values for rasterization
        role_filter = 'PRED' if 'Prediction' in layer_name else 'TRAIN'
        target_names = [name for name, role in APP.get('poly_roles', {}).items() if role == role_filter]
        if not target_names:
            with out_log: print(f"    No polygons found for role {role_filter}")
            return

        shapes_for_rasterize = []
        all_geoms = []
        for name in target_names:
            geoms = RAW_DATA['poly_dict'][name]['geoms']
            poly_uids = RAW_DATA['poly_dict'][name].get('poly_uids', [])
            for idx, geom in enumerate(geoms):
                poly_uid = poly_uids[idx] if idx < len(poly_uids) else None
                if poly_uid is None:
                    continue
                row = df_results[df_results['poly_uid'] == poly_uid]
                if row.empty:
                    continue
                val = row[val_col].iloc[0]
                if pd.isna(val):
                    continue
                geom_repaired = repair_geometry(geom)
                if geom_repaired is not None:
                    shapes_for_rasterize.append((geom_repaired, float(val)))
                    all_geoms.append(geom_repaired)

        if not shapes_for_rasterize:
            with out_log: print(f"    No matching geometries for '{layer_name}'")
            return

        # Set up colormap and normalization
        if is_categorical:
            vmin, vmax = 0, len(palette) - 1
            cmap = mcolors.ListedColormap(palette)
            norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        else:
            if dynamic_scale:
                vals = [v for _, v in shapes_for_rasterize]
                vmin, vmax = min(vals), max(vals)
                if vmin == vmax:
                    vmax = vmin + 1e-6
            else:
                vmin, vmax = 0.0, 1.0
            cmap = mcolors.LinearSegmentedColormap.from_list("custom", palette)
            norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

        # Rasterize
        local_gdf = gpd.GeoDataFrame(geometry=all_geoms, crs="EPSG:4326")
        minx, miny, maxx, maxy = local_gdf.total_bounds
        res = 0.0002
        width = int((maxx - minx) / res)
        height = int((maxy - miny) / res)
        max_pixels = 10000
        if width > max_pixels or height > max_pixels:
            scale = max(width, height) / max_pixels
            width = int(width / scale)
            height = int(height / scale)
        transform = rasterio.transform.from_bounds(minx, miny, maxx, maxy, width, height)
        raster = rasterio.features.rasterize(
            shapes=shapes_for_rasterize, out_shape=(height, width),
            transform=transform, fill=-9999.0, dtype=np.float32, all_touched=True
        )
        masked_raster = np.ma.masked_where(raster == -9999.0, raster)
        rgba_array = cmap(norm(masked_raster))
        rgba_array[masked_raster.mask] = [0, 0, 0, 0]

        # Save and display
        img = Image.fromarray((rgba_array * 255).astype(np.uint8))
        buffered = BytesIO()
        img.save(buffered, format="PNG")

        img.save(png_path, format="PNG")
        bounds = [(miny, minx), (maxy, maxx)]
        with open(bounds_path, "w") as b_file:
            json.dump(bounds, b_file)

        img_str = base64.b64encode(buffered.getvalue()).decode()
        image_url = f"data:image/png;base64,{img_str}"
        image_layer = ImageOverlay(url=image_url, bounds=bounds, name=layer_name, opacity=0.85)
        APP['map'].add_layer(image_layer)
        with out_log: print(f"    Layer '{layer_name}' cached to Drive.")

    except Exception as e:
        with out_log: print(f"Local Image Rendering Error: {e}")


# -----------------------------------------------------------------------------
# UI COMPONENTS (floating panels, legend, log)
# -----------------------------------------------------------------------------
def show_floating_panel(content_list, title, panel_key, width='600px'):
    """Adds a floating panel (toggleable with minus/plus) to the map."""
    if APP['panels'].get(panel_key):
        try: APP['map'].remove_control(APP['panels'][panel_key])
        except: pass
    btn_toggle = widgets.Button(icon='minus', layout=widgets.Layout(width='30px'), button_style='')
    content_box = widgets.VBox(content_list)
    def toggle_panel(b):
        content_box.layout.display = 'none' if content_box.layout.display != 'none' else 'flex'
        btn_toggle.icon = 'plus' if content_box.layout.display == 'none' else 'minus'
    btn_toggle.on_click(toggle_panel)
    panel = widgets.VBox([
        widgets.HBox([widgets.HTML(f'<span style="color: black; font-weight:bold;">{title}</span>'), btn_toggle],
                     layout=widgets.Layout(justify_content='space-between', border_bottom='1px solid #ddd')),
        content_box
    ], layout=widgets.Layout(width=width, background_color='white', padding='10px', max_height='550px', overflow_y='auto'))
    control = WidgetControl(widget=panel, position='bottomright')
    APP['map'].add_control(control)
    APP['panels'][panel_key] = control

def create_legend_widget():
    """Builds the legend for probability and confusion maps."""
    gradient = widgets.HTML(f'<div style="width:200px; height:12px; background:linear-gradient(to right,{", ".join(VIS_PALETTE)}); border:1px solid #999; border-radius:2px;"></div>')
    items = [widgets.HTML(f'<div style="display:flex; align-items:center; margin:4px 0;"><div style="width:14px; height:14px; border-radius:3px; background-color:{col}; margin-right:8px;"></div><span style="color: black !important; font-size:12px;">{txt}</span></div>')
             for col, txt in zip(PALETTE_CONFUSION, ["False Positive", "True Negative", "False Negative", "True Positive"])]
    return widgets.VBox([
        widgets.HTML("<div style='color: black !important; font-weight: bold; margin-bottom: 5px;'>Probability Map</div>"),
        gradient,
        widgets.HTML("<div style='color: black !important; font-weight: bold; margin-top: 10px; margin-bottom: 5px;'>Confusion Map</div>"),
        *items
    ], layout=widgets.Layout(padding='15px', background_color='rgba(255, 255, 255, 0.95)', border='1px solid #ccc', border_radius='8px', box_shadow='0 4px 6px rgba(0,0,0,0.1)'))


# Output widget for logs
out_log = widgets.Output(layout={'border': '1px solid #e0e0e0', 'height': '200px', 'overflow_y': 'scroll', 'border_radius': '5px'})
download_area = widgets.VBox()

# CSS for custom buttons
map_css = """<style>
    .custom-btn {
        background-color: #00BFFF !important;
        color: white !important; border: none !important; border-radius: 6px !important;
        box-shadow: 0 3px 6px rgba(0,0,0,0.16) !important; font-weight: 600 !important; font-size: 13px !important;
        transition: all 0.2s ease !important;
    }
    .custom-btn:hover {
        box-shadow: 0 5px 12px rgba(0,0,0,0.3) !important; transform: translateY(-1px) !important;
        background-color: #009acd !important;
    }
    .custom-btn:disabled {
        background-color: #cccccc !important;
        color: #666666 !important;
        box-shadow: none !important;
        cursor: not-allowed !important;
    }
    .widget-datepicker > div { border: 1px solid #ced4da !important; border-radius: 5px !important; }
</style>"""
css_widget_1, css_widget_2 = widgets.HTML(map_css), widgets.HTML(map_css)

# Input selectors
w_train_selector = widgets.Dropdown(description='Training:', layout=widgets.Layout(width='100%'), disabled=True)
w_pred_selector = widgets.Dropdown(description='Prediction:', layout=widgets.Layout(width='100%'), disabled=True)
w_points_selector = widgets.Dropdown(description='Points:', layout=widgets.Layout(width='100%'), disabled=True)

# Buttons
btn_style = widgets.ButtonStyle()
btn_upload_start = widgets.Button(description='Start Upload', style=btn_style, layout=widgets.Layout(width='100%', height='35px')).add_class('custom-btn')
btn_clear_cache = widgets.Button(description='Clear Cache', style=btn_style, layout=widgets.Layout(width='100%', height='35px')).add_class('custom-btn')
btn_calc_morph = widgets.Button(description='Calc Morphometry', style=btn_style, layout=widgets.Layout(width='100%', height='35px')).add_class('custom-btn')
btn_run_analysis = widgets.Button(description="Run Analysis", style=btn_style, layout=widgets.Layout(width='100%', height='35px')).add_class('custom-btn')
for btn in [btn_upload_start, btn_clear_cache, btn_calc_morph, btn_run_analysis]:
    btn._click_handlers.callbacks = []

tomorrow_date = datetime.date.today() + datetime.timedelta(days=1)
b_calib = widgets.Button(description="Run Calibration", layout=widgets.Layout(width='100%', height='38px')).add_class('custom-btn')
b_valid = widgets.Button(description="Run Validation", layout=widgets.Layout(width='100%', height='38px')).add_class('custom-btn')
w_pred_date = widgets.DatePicker(description='Date:', value=tomorrow_date, layout=widgets.Layout(width='100%'))
b_pred  = widgets.Button(description="Run Prediction", layout=widgets.Layout(width='100%', height='38px')).add_class('custom-btn')
for btn in [b_calib, b_valid, b_pred]:
    btn._click_handlers.callbacks = []


def set_ui_state(disabled=True):
    """Enable/disable all action buttons during processing."""
    b_calib.disabled = disabled
    b_valid.disabled = disabled
    b_pred.disabled = disabled
    btn_calc_morph.disabled = disabled
    btn_upload_start.disabled = disabled
    btn_clear_cache.disabled = disabled


# -----------------------------------------------------------------------------
# FILE UPLOAD AND CACHE MANAGEMENT
# -----------------------------------------------------------------------------
def process_input_files():
    """Load vector files from cache, reproject if needed, and display on map."""
    RAW_DATA['poly_dict'] = {}
    RAW_DATA['points_dict'] = {}
    RAW_DATA['poly_geoms'] = {}
    RAW_DATA['poly_gdfs'] = {}
    RAW_DATA['pts_gdf'] = None

    if len(APP['map'].layers) > 1:
        APP['map'].layers = APP['map'].layers[:1]

    file_list = sorted(glob.glob(os.path.join(INPUT_CACHE_DIR, '*')))
    if not file_list:
        print("No cached files found.")
        return

    for v_file in file_list:
        if not v_file.endswith(('.geojson', '.gpkg', '.shp')):
            continue
        try:
            with fiona.open(v_file) as src:
                crs = src.crs
                geom_type = src.schema['geometry']
                features = list(src)
                file_mb = os.path.getsize(v_file) / (1024 * 1024)

                if 'Polygon' in geom_type or 'MultiPolygon' in geom_type:
                    fname = os.path.basename(v_file)
                    print(f" Found Area: {fname} ({file_mb:.2f} MB)")
                    shapely_geoms = []
                    local_props = []
                    poly_uids = []

                    for idx, feat in enumerate(features):
                        geom = shape(feat['geometry'])
                        if crs and src.crs.to_epsg() != 4326:
                            project = partial(pyproj.transform, pyproj.Proj(crs), pyproj.Proj(4326))
                            geom = shapely_transform(project, geom)
                        geom = repair_geometry(geom)
                        if geom is None:
                            poly_uids.append(f"INVALID_{idx}")
                            continue
                        shapely_geoms.append(geom)
                        props = dict(feat['properties']) if feat['properties'] else {}
                        local_props.append(props)

                        try:
                            rep_point = geom.representative_point()
                            lon = rep_point.x
                            lat = rep_point.y
                            poly_uid = f"{round(lon, 6)}_{round(lat, 6)}"
                        except:
                            poly_uid = f"ERROR_{idx}"
                        poly_uids.append(poly_uid)

                    if not shapely_geoms:
                        continue

                    RAW_DATA['poly_dict'][fname] = {
                        'geoms': shapely_geoms,
                        'props': pd.DataFrame(local_props),
                        'poly_uids': poly_uids
                    }
                    RAW_DATA['poly_geoms'][fname] = shapely_geoms
                    RAW_DATA['poly_gdfs'][fname] = pd.DataFrame(local_props)

                    print(f"    Rasterizing polygons")
                    local_gdf = gpd.GeoDataFrame(geometry=shapely_geoms, crs="EPSG:4326")
                    minx, miny, maxx, maxy = local_gdf.total_bounds
                    res = 0.0002
                    width = int((maxx - minx) / res)
                    height = int((maxy - miny) / res)
                    max_pixels = 10000
                    if width > max_pixels or height > max_pixels:
                        scale = max(width, height) / max_pixels
                        width = int(width / scale)
                        height = int(height / scale)
                    transform_mat = rasterio.transform.from_bounds(minx, miny, maxx, maxy, width, height)
                    shapes_for_rasterize = [(geom, 1) for geom in shapely_geoms]
                    raster = rasterio.features.rasterize(
                        shapes_for_rasterize, out_shape=(height, width),
                        transform=transform_mat, fill=0, dtype=np.uint8, all_touched=True
                    )
                    rgba_array = np.zeros((height, width, 4), dtype=np.uint8)
                    rgba_array[raster == 1] = [255, 0, 0, 90]
                    img = Image.fromarray(rgba_array)
                    buffered = BytesIO()
                    img.save(buffered, format="PNG")
                    img_str = base64.b64encode(buffered.getvalue()).decode()
                    APP['map'].add_layer(
                        ImageOverlay(
                            url=f"data:image/png;base64,{img_str}",
                            bounds=[(miny, minx), (maxy, maxx)],
                            name=f'Area: {fname} (Rasterized)',
                            opacity=0.85
                        )
                    )
                    APP['map'].center = [(miny+maxy)/2, (minx+maxx)/2]
                    APP['map'].zoom = 9

                elif 'Point' in geom_type or 'MultiPoint' in geom_type:
                    fname = os.path.basename(v_file)
                    print(f" Found Landslides: {fname}")
                    ee_features = []
                    pts_geoms = []
                    pts_props = []

                    for feat in features:
                        geom = shape(feat['geometry'])
                        if crs and src.crs.to_epsg() != 4326:
                            project = partial(pyproj.transform, pyproj.Proj(crs), pyproj.Proj(4326))
                            geom = shapely_transform(project, geom)
                        props = dict(feat['properties']) if feat['properties'] else {}
                        ee_features.append(ee.Feature(ee.Geometry(mapping(geom)), props))
                        pts_geoms.append(geom)
                        pts_props.append(props)

                    if not ee_features:
                        continue

                    micro_pts = [ee.FeatureCollection(ee_features[i:i+1000]) for i in range(0, len(ee_features), 1000)]
                    ee_fc = ee.FeatureCollection(ee.List(micro_pts)).flatten()
                    gdf = gpd.GeoDataFrame(pts_props, geometry=pts_geoms, crs="EPSG:4326")

                    RAW_DATA['points_dict'][fname] = {
                        'ee_fc': ee_fc,
                        'gdf': gdf,
                        'props': pd.DataFrame(pts_props)
                    }
                    RAW_DATA['pts_gdf'] = gdf

                    try:
                        print(f"    Rendering {len(pts_geoms)} points")
                        geo_layer = GeoData(
                            geo_dataframe=gdf,
                            name=f'Points: {fname}',
                            style={'color': 'black', 'fillColor': 'black', 'weight': 1},
                            point_style={'radius': 3, 'color': 'black', 'fillColor': 'black',
                                         'fillOpacity': 1, 'weight': 1}
                        )
                        APP['map'].add_layer(geo_layer)
                    except Exception as e:
                        print(f"    Error rendering points: {e}")

        except Exception as e:
            print(f"Skip {os.path.basename(v_file)}: {e}")

    poly_names = list(RAW_DATA['poly_dict'].keys())
    if poly_names:
        w_train_selector.options = poly_names
        w_pred_selector.options = poly_names
        w_train_selector.value = poly_names[0] if len(poly_names) > 0 else None
        w_pred_selector.value = poly_names[1] if len(poly_names) > 1 else poly_names[0]
        w_train_selector.disabled = False
        w_pred_selector.disabled = False
    else:
        w_train_selector.disabled = True
        w_pred_selector.disabled = True

    point_names = list(RAW_DATA['points_dict'].keys())
    if point_names:
        w_points_selector.options = point_names
        w_points_selector.value = point_names[0]
        w_points_selector.disabled = False
    else:
        w_points_selector.disabled = True


def on_upload_start(b):
    """Upload vector files to Drive cache and process them."""
    keep_colab_alive()
    ensure_drive_mounted(out_log)
    out_log.clear_output()
    download_area.children = []
    with out_log:
        if colab_files is None:
            print("Google Colab files API not available.")
            return
        cached_files = [f for f in os.listdir(INPUT_CACHE_DIR) if f.endswith(('.geojson', '.gpkg', '.shp'))]
        if cached_files:
            print("Files already cached. No upload needed.")
            process_input_files()
        else:
            print("Cache is empty. Please upload vector files...")
            uploaded_files = colab_files.upload()
            if uploaded_files:
                for fname, content in uploaded_files.items():
                    with open(os.path.join(INPUT_CACHE_DIR, fname), 'wb') as f:
                        f.write(content)
                print("Upload complete. Processing files.")
                process_input_files()


def on_clear_cache_click(b):
    """Remove all cached files, reset APP and RAW_DATA."""
    keep_colab_alive()
    ensure_drive_mounted(out_log)
    out_log.clear_output()
    with out_log:
        try:
            print("INITIATING DEEP CACHE CLEAN...")
            if os.path.exists(INPUT_CACHE_DIR):
                shutil.rmtree(INPUT_CACHE_DIR)
            os.makedirs(INPUT_CACHE_DIR, exist_ok=True)

            cache_dir = os.path.join(RESULTS_DIR, 'Cache')
            if os.path.exists(cache_dir):
                shutil.rmtree(cache_dir)
            os.makedirs(cache_dir, exist_ok=True)

            if os.path.exists(RASTER_DIR):
                shutil.rmtree(RASTER_DIR)
            os.makedirs(RASTER_DIR, exist_ok=True)

            rain_cache = os.path.join(RESULTS_DIR, 'Rain_Cache')
            if os.path.exists(rain_cache):
                shutil.rmtree(rain_cache)
            os.makedirs(rain_cache, exist_ok=True)

            for f in glob.glob(os.path.join(RESULTS_DIR, "*.joblib")) + \
                     glob.glob(os.path.join(RESULTS_DIR, "*.csv")) + \
                     glob.glob(os.path.join(RESULTS_DIR, "*.progress")) + \
                     glob.glob(os.path.join(RESULTS_DIR, "*.geojson")):
                try:
                    os.remove(f)
                except:
                    pass

            APP['training_df'] = None
            APP['prediction_df'] = None
            APP['model'] = None
            APP['final_predictors'] = []
            APP['original_predictors'] = []
            APP['current_hash'] = 'default'
            APP['dummies_map'] = None

            if len(APP['map'].layers) > 1:
                APP['map'].layers = APP['map'].layers[:1]

            w_train_selector.options = []
            w_pred_selector.options = []
            w_points_selector.options = []
            w_train_selector.disabled = True
            w_pred_selector.disabled = True
            w_points_selector.disabled = True

            RAW_DATA['poly_dict'] = {}
            RAW_DATA['points_dict'] = {}
            RAW_DATA['poly_geoms'] = {}
            RAW_DATA['poly_gdfs'] = {}
            RAW_DATA['pts_gdf'] = None

            print("COMPLETE CACHE CLEARED: Inputs, matrices, models, and predictions wiped.")
            print("System fully reset for a new study area.")
        except Exception as e:
            print(f"Error clearing cache: {e}")


# -----------------------------------------------------------------------------
# MORPHOMETRY EXTRACTION (raster values at point locations)
# -----------------------------------------------------------------------------
def on_morphometry_click(b):
    """Extract static morphometric and spectral covariates from GEE for training/prediction polygons."""
    set_ui_state(True)
    keep_colab_alive()
    ensure_drive_mounted(out_log)
    out_log.clear_output()
    class LogWriter:
        def write(self, msg):
            with out_log: print(msg)
    wrapper = LogWriter()

    try:
        train_name = w_train_selector.value
        pred_name = w_pred_selector.value
        points_name = w_points_selector.value

        if not train_name or not pred_name or not points_name:
            with out_log:
                print("ERROR: Please select Training area, Prediction area, and Points file.")
            set_ui_state(False)
            return

        poly_roles = {}
        for name in RAW_DATA['poly_dict']:
            if name == train_name:
                poly_roles[name] = 'TRAIN'
            elif name == pred_name:
                poly_roles[name] = 'PRED'
            else:
                poly_roles[name] = 'IGNORE'
        APP['poly_roles'] = poly_roles

        points_data = RAW_DATA['points_dict'][points_name]
        points_fc = points_data['ee_fc']
        points_gdf = points_data['gdf']
        RAW_DATA['pts_gdf'] = points_gdf

        cache_dir = '/content/drive/MyDrive/Morfometry_Results/Cache'
        os.makedirs(cache_dir, exist_ok=True)

        train_hash = f"{train_name.split('.')[0]}_{pred_name.split('.')[0]}_{points_name.split('.')[0]}"
        APP['current_hash'] = train_hash
        master_train_csv = os.path.join(cache_dir, f"MASTER_TRAIN_{train_hash}.csv")

        train_list, pred_list = [], []

        with out_log:
            print(f"STARTING MORPHOMETRY EXTRACTION...")
            print(f"    Training area: {train_name}")
            print(f"    Prediction area: {pred_name}")
            print(f"    Points file: {points_name}")

            if os.path.exists(master_train_csv):
                print(f"    Loading Master matrix from Drive using Polars...")
                APP['training_df'] = pd.read_csv(master_train_csv, low_memory=False)

                for name, role in poly_roles.items():
                    if role == 'PRED':
                        pred_cache_file = os.path.join(cache_dir, f"{name}_{role}_static.csv")
                        if os.path.exists(pred_cache_file):
                            pred_list.append(pd.read_csv(pred_cache_file, low_memory=False))
                if pred_list:
                    APP['prediction_df'] = pd.concat(pred_list, ignore_index=True)
                elif train_name == pred_name:
                    train_cache_file = os.path.join(cache_dir, f"{train_name}_TRAIN_static.csv")
                    if os.path.exists(train_cache_file):
                        pred_base = pd.read_csv(train_cache_file, low_memory=False)
                        cols_to_drop = [c for c in ['P/A', 'event_date', 'proc_date', 'formatted_', 'date', 'system:index', 'matched_point', 'geometry'] if c in pred_base.columns]
                        APP['prediction_df'] = pred_base.drop(columns=cols_to_drop)
            else:
                for name, role in poly_roles.items():
                    if role == 'IGNORE':
                        continue
                    print(f"\n--- PROCESSING AREA: {name} ({role}) ---")

                    stored_poly_uids = RAW_DATA['poly_dict'][name].get('poly_uids', [])
                    local_props = RAW_DATA['poly_dict'][name]['props'].copy()
                    shapely_geoms = RAW_DATA['poly_dict'][name]['geoms']
                    local_gdf = gpd.GeoDataFrame(local_props, geometry=shapely_geoms, crs="EPSG:4326")

                    local_gdf['poly_uid'] = stored_poly_uids[:len(local_gdf)]

                    def get_lon_from_uid(uid):
                        try:
                            return float(str(uid).split('_')[0])
                        except:
                            return 0.0
                    def get_lat_from_uid(uid):
                        try:
                            return float(str(uid).split('_')[1])
                        except:
                            return 0.0
                    local_gdf['lon'] = local_gdf['poly_uid'].apply(get_lon_from_uid)
                    local_gdf['lat'] = local_gdf['poly_uid'].apply(get_lat_from_uid)

                    cache_file = os.path.join(cache_dir, f"{name}_{role}_static.csv")
                    area_dfs = []
                    start_idx = 0

                    if os.path.exists(cache_file):
                        res = pd.read_csv(cache_file)
                        if len(res) >= len(local_gdf):
                            print(f"    Cache found ({len(res)} rows). Loading...")
                            if role == 'TRAIN':
                                train_list.append(res)
                            else:
                                pred_list.append(res)
                            continue
                        else:
                            print(f"    Partial cache ({len(res)} rows). Resuming...")
                            area_dfs.append(res)
                            start_idx = len(res)

                    if role == 'TRAIN':
                        pa_col = next((c for c in local_gdf.columns if c.upper() == 'GRIDCODE'), None)
                        if not pa_col:
                            pa_col = next((c for c in local_gdf.columns if c.upper() in ['P/A', 'PRESENCE', 'CLASS', 'TARGET', 'LANDSLIDE']), None)

                        if pa_col:
                            print(f"    Using target column '{pa_col}'")
                            local_gdf['P/A'] = local_gdf[pa_col].astype(int)
                            if 'event_date' not in local_gdf.columns:
                                local_gdf['event_date'] = 0
                        elif points_gdf is not None:
                            pts = points_gdf.copy()
                            date_col = DATE_COLUMN if DATE_COLUMN in pts.columns else pts.columns[0]
                            with warnings.catch_warnings():
                                warnings.simplefilter("ignore")
                                joined = gpd.sjoin(local_gdf, pts, how='left', predicate='intersects')
                            joined = joined[~joined.index.duplicated(keep='first')]
                            local_gdf['P/A'] = joined['index_right'].notna().astype(int).values
                            local_gdf['event_date'] = joined[date_col].fillna(0).values
                        else:
                            local_gdf['P/A'] = 0
                            local_gdf['event_date'] = 0

                        print(f"    Total training rows: {len(local_gdf)}")
                    else:
                        local_gdf['P/A'] = 0
                        local_gdf['event_date'] = 0

                    points_df_to_extract = local_gdf[['poly_uid', 'P/A', 'event_date', 'lon', 'lat']].copy()
                    RAM_SAFE_STEP = 1500

                    for ram_start in range(start_idx, len(points_df_to_extract), RAM_SAFE_STEP):
                        ram_end = min(ram_start + RAM_SAFE_STEP, len(points_df_to_extract))
                        print(f"    Extracting covariates {ram_start} to {ram_end}...")
                        sub_df = points_df_to_extract.iloc[ram_start:ram_end]
                        try:
                            # FIX: GEE estrae solo le metriche, ignora i categorici (lista vuota [])
                            res = extract_covariates_fast(sub_df, wrapper, SELECTED_METRICS_USER, [])
                            if res is not None and not res.empty:
                                area_dfs.append(res)
                                temp_df = pd.concat(area_dfs, ignore_index=True)
                                temp_df.to_csv(cache_file, index=False)
                                wrapper.write(f"      Checkpoint: {len(temp_df)}/{len(local_gdf)} rows saved.")
                        except Exception as e:
                            wrapper.write(f"     Extraction error: {e}")
                        gc.collect()

                    if area_dfs:
                        full_df = pd.concat(area_dfs, ignore_index=True)

                        # FIX: RECUPERO DINAMICO: aggiunge i campi categorici presenti nel file vettoriale
                        cat_cols = [c for c in CATEGORICAL_METRICS if c in local_gdf.columns]
                        if cat_cols:
                            full_df = pd.merge(full_df, local_gdf[['poly_uid'] + cat_cols], on='poly_uid', how='left')

                        if role == 'TRAIN':
                            train_list.append(full_df)
                        else:
                            pred_list.append(full_df)

                if train_list:
                    print("BUILDING SPATIO-TEMPORAL MATRIX...")
                    base_morph = pd.concat(train_list, ignore_index=True).drop_duplicates(subset=['poly_uid'])
                    cols_to_drop = [c for c in ['P/A', 'event_date', 'proc_date', 'formatted_', 'date',
                                                'system:index', 'matched_point', 'geometry'] if c in base_morph.columns]
                    base_morph = base_morph.drop(columns=cols_to_drop)

                    unique_dates = []
                    if points_fc is not None:
                        date_list = points_fc.aggregate_array(DATE_COLUMN).getInfo()
                        unique_dates = sorted({str(d)[:10].replace('/','-') for d in date_list if d})
                    else:
                        unique_dates = sorted({str(d) for d in base_morph['event_date'].unique() if d != 0})

                    if unique_dates:
                        base_pl = pl.from_pandas(base_morph)
                        dates_pl = pl.DataFrame({'event_date': unique_dates})
                        cross_pl = base_pl.join(dates_pl, how='cross')

                        positive_pairs = pd.concat(train_list, ignore_index=True)[['poly_uid','event_date', 'P/A']]
                        positive_pairs = positive_pairs[positive_pairs['P/A'] == 1][['poly_uid','event_date']].drop_duplicates()
                        positive_pairs['event_date'] = positive_pairs['event_date'].astype(str).str[:10].str.replace('/', '-')

                        positive_pl = pl.from_pandas(positive_pairs).with_columns(pl.lit(1).alias('P/A'))
                        cross_pl = cross_pl.join(positive_pl, on=['poly_uid', 'event_date'], how='left') \
                                         .with_columns(pl.col('P/A').fill_null(0))

                        cross_df = cross_pl.to_pandas()
                    else:
                        cross_df = base_morph

                    print(f"    Matrix created: {len(cross_df)} rows")
                    APP['training_df'] = cross_df
                    APP['training_df'] = download_rainfall_data(out_log)

                    print(f"    Saving master CSV...")
                    APP['training_df'].to_csv(master_train_csv, index=False)

                if pred_list:
                    APP['prediction_df'] = pd.concat(pred_list, ignore_index=True)
                elif train_name == pred_name and train_list:
                    print("    Using Training area as Prediction area too...")
                    APP['prediction_df'] = base_morph.copy()

            print("\n MORPHOMETRY COMPLETED.")

    except Exception as e:
        with out_log: print(f"CRITICAL ERROR: {e}")
    finally:
        set_ui_state(False)


# -----------------------------------------------------------------------------
# CALIBRATION (model training and best rainfall window selection)
# -----------------------------------------------------------------------------
def on_calib_click(b):
    """Train Random Forest on static + dynamic rainfall, select best window (7/14 days)."""
    set_ui_state(True)
    keep_colab_alive()
    ensure_drive_mounted(out_log)
    out_log.clear_output()
    with out_log:
        try:
            print("=" * 60)
            print("  PRE-CALIBRATION DATA INTEGRITY CHECK")
            print("=" * 60)

            if APP['training_df'] is None:
                print("ERROR: training_df is None. Run 'Calc Morphometry' first.")
                set_ui_state(False)
                return

            df = APP['training_df']
            print(f"training_df loaded: {len(df)} rows")

            required_cols = ['poly_uid', 'P/A']
            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                print(f"ERROR: Missing required columns: {missing}")
                set_ui_state(False)
                return
            print("Required columns present: poly_uid, P/A")

            pos = (df['P/A'] == 1).sum()
            neg = (df['P/A'] == 0).sum()
            print(f"   Positive: {pos} | Negative: {neg}")
            if pos == 0:
                print("ERROR: No positive samples (P/A = 1). Add landslide points.")
                set_ui_state(False)
                return
            if neg == 0:
                print("ERROR: No negative samples (P/A = 0). Add non-landslide polygons.")
                set_ui_state(False)
                return
            print("Class balance OK.")

            for days in [7, 14]:
                col = f'Rn{days}_m'
                if col not in df.columns:
                    print(f"ERROR: '{col}' missing. Rainfall extraction failed.")
                    set_ui_state(False)
                    return
                else:
                    nan_count = df[col].isna().sum()
                    if nan_count > 0:
                        print(f"   {col} has {nan_count} NaNs (will be filled with 0)")
                    else:
                        print(f"   {col} present, no NaNs")

            excluded = ['P/A', 'event_date', 'poly_uid', 'system:index', 'date_key',
                        'proc_date', '.geo', 'matched_point', 'date', 'formatted_']
            static_cols = [c for c in df.columns if c not in excluded and c not in ['Rn7_m', 'Rn14_m']]
            if not static_cols:
                print("ERROR: No static predictors found. Check morphometry extraction.")
                set_ui_state(False)
                return
            print(f"Static predictors found: {len(static_cols)}")

            if 'event_date' in df.columns:
                sample_dates = df['event_date'].dropna().unique()[:3]
                print(f"   Example dates: {sample_dates}")
            else:
                print("   No 'event_date' column – will use cross-join without dates.")

            print("=" * 60)
            print("  ALL CHECKS PASSED – STARTING CALIBRATION")
            print("=" * 60)

            train_hash = APP.get('current_hash', 'default')
            model_cache_path = os.path.join(RESULTS_DIR, f"MASTER_MODEL_{train_hash}.joblib")
            calib_csv_path = os.path.join(RESULTS_DIR, f"cache_calibration_{train_hash}.csv")
            calib_prog = calib_csv_path + ".progress"

            skip_training = False
            start_chunk = 0

            if os.path.exists(model_cache_path) and os.path.exists(calib_csv_path):
                print("    Model found for this area. Loading from Drive...")
                cached_data = joblib.load(model_cache_path)
                APP['model'] = cached_data['model']
                APP['final_predictors'] = cached_data['predictors']
                APP['original_predictors'] = cached_data.get('original_predictors', [])
                APP['best_days'] = cached_data.get('best_days', 7)
                APP['dummies_map'] = cached_data.get('dummies_map', None)
                best_m = cached_data['metrics']
                best_days = cached_data['best_days']
                y_balanced = cached_data['y_balanced']
                skip_training = True

                if os.path.exists(calib_prog):
                    with open(calib_prog, 'r') as f:
                        start_chunk = int(f.read().strip())
                    print(f"    Resuming predictions from row {start_chunk}...")
                else:
                    print("    Full calibration CSV already exists.")
                    start_chunk = len(APP['training_df'])
            elif os.path.exists(calib_csv_path):
                os.remove(calib_csv_path)

            if start_chunk == 0:
                sort_cols = ['poly_uid', 'event_date'] if 'event_date' in df.columns else ['poly_uid']
                df.sort_values(by=sort_cols, inplace=True)
                df.reset_index(drop=True, inplace=True)

            if not skip_training:
                excl = ['P/A', 'event_date', 'poly_uid', 'system:index', 'date_key', 'proc_date', '.geo', 'matched_point', 'date', 'formatted_']
                base_preds = [c for c in df.columns if c not in excl and c not in ['Rn7_m', 'Rn14_m']]

                cat_cols = [c for c in base_preds if c in CATEGORICAL_METRICS]
                if cat_cols:
                    print(f"    Categorical predictors found: {cat_cols}")
                    sample_idx = np.random.choice(len(df), min(50000, len(df)), replace=False)
                    _, _, dummies_map = encode_categoricals(
                        df.iloc[sample_idx][base_preds], base_preds, cat_cols, dummies_map=None
                    )
                    APP['dummies_map'] = dummies_map
                else:
                    APP['dummies_map'] = None

                print("Model calibration...")

                MAX_SAMPLES = 400000
                indices = np.arange(len(df))
                if len(df) > MAX_SAMPLES:
                    print(f"    Subsampling {MAX_SAMPLES} rows for training")
                    from sklearn.model_selection import train_test_split
                    train_idx, _ = train_test_split(indices, train_size=MAX_SAMPLES, stratify=df['P/A'], random_state=42)
                else:
                    train_idx = indices

                y_balanced = df['P/A'].values[train_idx].astype(int)

                best_auc, best_days, best_model, best_encoded_preds, best_m = 0, None, None, None, None
                best_original_preds = None
                best_dummies_map = None

                for days in [7, 14]:
                    rain_col = f'Rn{days}_m'
                    if rain_col not in df.columns: continue
                    original_preds = base_preds + [rain_col]

                    X_train, encoded_preds, dummies_map = encode_categoricals(
                        df.iloc[train_idx][original_preds],
                        original_preds,
                        cat_cols if cat_cols else [],
                        dummies_map=APP.get('dummies_map')
                    )
                    X_train = X_train.fillna(0)

                    rf = Pipeline([("scaler", StandardScaler()), ("random_forest", RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=2))])
                    rf.fit(X_train, y_balanced)

                    probs_balanced = rf.predict_proba(X_train)[:, 1]
                    m = calculate_advanced_metrics_polars(y_balanced, probs_balanced)
                    print(f"   > {days}-day rainfall: AUC = {m['auc']:.4f}")

                    if m['auc'] > best_auc:
                        best_auc, best_days, best_model, best_encoded_preds, best_m = m['auc'], days, rf, encoded_preds, m
                        best_original_preds = original_preds
                        best_dummies_map = dummies_map

                    del rf, X_train
                    gc.collect()

                if best_model is None:
                    print("ERROR: No valid model could be trained. Check rainfall data.")
                    set_ui_state(False)
                    return

                print(f"Best window: {best_days} days (AUC: {best_auc:.4f})")
                APP['model'] = best_model
                APP['final_predictors'] = best_encoded_preds
                APP['original_predictors'] = best_original_preds
                APP['best_days'] = best_days
                APP['dummies_map'] = best_dummies_map

                joblib.dump({
                    'model': best_model,
                    'predictors': best_encoded_preds,
                    'original_predictors': best_original_preds,
                    'metrics': best_m,
                    'best_days': best_days,
                    'y_balanced': y_balanced,
                    'dummies_map': best_dummies_map
                }, model_cache_path)

            if start_chunk < len(df):
                print("Applying predictions to full matrix...")
                CHUNK_PRED = 100000
                cat_cols = [c for c in APP['original_predictors'] if c in CATEGORICAL_METRICS]
                for i in range(start_chunk, len(df), CHUNK_PRED):
                    end_idx = min(i + CHUNK_PRED, len(df))
                    print(f"      Predicting rows {i} to {end_idx}...")

                    sub_df = df.iloc[i:end_idx].copy()
                    X_chunk, _, _ = encode_categoricals(
                        sub_df[APP['original_predictors']],
                        APP['original_predictors'],
                        cat_cols,
                        dummies_map=APP.get('dummies_map')
                    )
                    X_chunk = X_chunk.fillna(0)
                    probs = APP['model'].predict_proba(X_chunk)[:, 1]

                    sub_df['Susceptibility_Prob'] = probs
                    sub_df['Predicted_Class'] = (probs >= best_m['best_thresh']).astype(int)

                    sub_df.to_csv(calib_csv_path, index=False, mode='w' if i == 0 else 'a', header=(i == 0))

                    with open(calib_prog, 'w') as f:
                        f.write(str(end_idx))

                    del sub_df, X_chunk, probs
                    gc.collect()

                if os.path.exists(calib_prog): os.remove(calib_prog)

            df_full_results = pd.read_csv(calib_csv_path, usecols=['poly_uid', 'Susceptibility_Prob', 'Predicted_Class', 'P/A'], low_memory=False)

        except Exception as e:
            print(f"Calibration error: {e}")
            import traceback
            print(traceback.format_exc())
            set_ui_state(False)
            return

    try:
        add_map_layer_geemap_native(df_full_results, 'Susceptibility_Prob', 'Calibration Map', VIS_PALETTE, dynamic_scale=False)

        print("    Calculating confusion map...")
        df_full_results['conf_class'] = 1
        p_class = df_full_results['Predicted_Class'].values
        t_class = df_full_results['P/A'].values
        df_full_results.loc[(p_class == 1) & (t_class == 0), 'conf_class'] = 0
        df_full_results.loc[(p_class == 0) & (t_class == 1), 'conf_class'] = 2
        df_full_results.loc[(p_class == 1) & (t_class == 1), 'conf_class'] = 3
        del p_class, t_class
        gc.collect()

        add_map_layer_geemap_native(df_full_results, 'conf_class', 'Confusion Calibration', PALETTE_CONFUSION, is_categorical=True, dynamic_scale=False)

        cmatrix = confusion_matrix(y_balanced, best_m['y_pred_opt'])
        fig = make_subplots(rows=2, cols=2, specs=[[{"colspan": 2}, None], [{}, {}]], subplot_titles=("Feature Importance", "Confusion Matrix", "ROC Curve"))
        # FIX: Pipeline attribute access
        feat_imp = pd.Series(APP['model'].named_steps['random_forest'].feature_importances_, index=APP['final_predictors']).sort_values().tail(10)
        fig.add_trace(go.Bar(x=feat_imp.values, y=feat_imp.index, orientation='h'), row=1, col=1)
        fig.add_trace(go.Heatmap(z=cmatrix, x=['P:0', 'P:1'], y=['T:0', 'T:1'], colorscale='Blues', text=[[str(v) for v in r] for r in cmatrix], texttemplate="%{text}", showscale=False), row=2, col=1)
        fig.update_yaxes(autorange="reversed", row=2, col=1)

        step = max(1, len(best_m['fpr']) // 100)
        fpr_plot = best_m['fpr'][::step]
        tpr_plot = best_m['tpr'][::step]
        fig.add_trace(go.Scatter(x=fpr_plot, y=tpr_plot, fill='tozeroy', name='ROC'), row=2, col=2)
        fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(dash='dash'), name='Random'), row=2, col=2)
        fig.update_layout(height=450, margin=dict(t=30, b=10, l=10, r=10), showlegend=False, template="plotly_white")

        out_plot = widgets.Output()
        with out_plot: display(go.FigureWidget(fig))

        calib_geojson_path = os.path.join(RESULTS_DIR, f"calibration_full_{train_hash}.geojson")
        if not os.path.exists(calib_geojson_path):
            save_results_to_geojson(df_full_results, calib_geojson_path, role='TRAIN', aggregate=False)
        else:
            with out_log: print(f"    Skipping: {os.path.basename(calib_geojson_path)} already exists.")

        calib_conf_geojson_path = os.path.join(RESULTS_DIR, f"calibration_confusion_{train_hash}.geojson")
        if not os.path.exists(calib_conf_geojson_path):
            save_confusion_geojson(df_full_results, calib_conf_geojson_path, role='TRAIN')
        else:
            with out_log: print(f"    Skipping: {os.path.basename(calib_conf_geojson_path)} already exists.")

        header = widgets.HBox([
            widgets.HTML(f"<div style='color: #d10e00; font-weight:bold; font-size: 13px;'>BEST MODEL: {best_days} Days Rainfall | ACC: {best_m['acc']:.3f} | AUC: {best_m['auc']:.3f}</div>"),
            widgets.HBox([
                create_save_to_drive_button(calib_geojson_path, f"calibration_full_{train_hash}", title="Save Full GeoJSON"),
                create_save_to_drive_button(calib_conf_geojson_path, f"calibration_confusion_{train_hash}", title="Save Confusion GeoJSON")
            ])
        ], layout=widgets.Layout(justify_content='space-between', align_items='center', margin='0 0 10px 0', border_bottom='1px solid #ccc', padding='5px 0'))

        show_floating_panel([header, out_plot], "Calibration Results", 'calib', width='650px')

        with out_log: print(">> Calibration completed successfully.")

    except Exception as e:
        with out_log: print(f"ERROR during result display: {e}")
        import traceback
        with out_log: print(traceback.format_exc())
    finally:
        set_ui_state(False)


# -----------------------------------------------------------------------------
# VALIDATION (cross‑validation out‑of‑fold predictions)
# -----------------------------------------------------------------------------
def on_valid_click(b):
    """Run 5-fold cross-validation and display out-of-fold predictions."""
    set_ui_state(True)
    keep_colab_alive()
    ensure_drive_mounted(out_log)
    out_log.clear_output()
    with out_log:
        try:
            if APP['model'] is None:
                print("ERROR: Run Calibration first!"); set_ui_state(False); return

            train_hash = APP.get('current_hash', 'default')
            valid_cache_joblib = os.path.join(RESULTS_DIR, f"cache_validation_{train_hash}.joblib")
            valid_csv_path = os.path.join(RESULTS_DIR, f"cache_validation_{train_hash}.csv")

            skip_cv = False
            m = None
            cmatrix_cv = None
            df_full_results = None

            if os.path.exists(valid_cache_joblib) and os.path.exists(valid_csv_path):
                print("    CV metrics and out-of-fold predictions found. Loading...")
                cached_data = joblib.load(valid_cache_joblib)
                m = cached_data['metrics']
                cmatrix_cv = cached_data['cmatrix_cv']
                skip_cv = True
                df_full_results = pd.read_csv(valid_csv_path, usecols=['poly_uid', 'Susceptibility_Prob', 'P/A'], low_memory=False)
                best_thresh = m['best_thresh']
                df_full_results['Predicted_Class'] = (df_full_results['Susceptibility_Prob'] >= best_thresh).astype(int)
            else:
                print("    Running 5-fold cross-validation...")
                df = APP['training_df']
                MAX_SAMPLES = 400000
                indices = np.arange(len(df))
                if len(df) > MAX_SAMPLES:
                    from sklearn.model_selection import train_test_split
                    train_idx, _ = train_test_split(indices, train_size=MAX_SAMPLES, stratify=df['P/A'], random_state=42)
                else:
                    train_idx = indices

                y_cv = df['P/A'].values[train_idx].astype(int)
                poly_uids_cv = df['poly_uid'].values[train_idx]

                cat_cols = [c for c in APP['original_predictors'] if c in CATEGORICAL_METRICS]
                X_cv, _, _ = encode_categoricals(
                    df.iloc[train_idx][APP['original_predictors']],
                    APP['original_predictors'],
                    cat_cols,
                    dummies_map=APP.get('dummies_map')
                )
                X_cv = X_cv.fillna(0)
                X_cv_arr = np.ascontiguousarray(X_cv.to_numpy(), dtype=np.float32)
                del X_cv
                gc.collect()

                print("    Computing Out-of-Fold predictions...")
                probs_cv = cross_val_predict(
                    APP['model'],
                    X_cv_arr, y_cv,
                    cv=StratifiedKFold(5, shuffle=True, random_state=42),
                    method='predict_proba',
                    n_jobs=2
                )[:, 1]

                m = calculate_advanced_metrics_polars(y_cv, probs_cv)
                cmatrix_cv = confusion_matrix(y_cv, m['y_pred_opt'])

                joblib.dump({'metrics': m, 'cmatrix_cv': cmatrix_cv}, valid_cache_joblib)

                print("    Saving Out-of-Fold predictions to disk...")
                df_valid_results = pd.DataFrame({
                    'poly_uid': poly_uids_cv,
                    'Susceptibility_Prob': probs_cv,
                    'P/A': y_cv,
                    'Predicted_Class': (probs_cv >= m['best_thresh']).astype(int)
                })
                df_valid_results.to_csv(valid_csv_path, index=False)

                df_full_results = df_valid_results.copy()

                del X_cv_arr, y_cv, probs_cv, df_valid_results
                gc.collect()
                print("    Cross-validation completed successfully.")

        except Exception as e:
            print(f"ERROR: {e}")
            set_ui_state(False)
            return

    try:
        add_map_layer_geemap_native(df_full_results, 'Susceptibility_Prob', 'Validation Map', VIS_PALETTE, dynamic_scale=False)

        print("    Calculating confusion map (validation, out-of-fold)...")
        df_full_results['conf_class'] = 1
        p_class = df_full_results['Predicted_Class'].values
        t_class = df_full_results['P/A'].values
        df_full_results.loc[(p_class == 1) & (t_class == 0), 'conf_class'] = 0
        df_full_results.loc[(p_class == 0) & (t_class == 1), 'conf_class'] = 2
        df_full_results.loc[(p_class == 1) & (t_class == 1), 'conf_class'] = 3
        del p_class, t_class
        gc.collect()

        add_map_layer_geemap_native(df_full_results, 'conf_class', 'Confusion Validation', PALETTE_CONFUSION, is_categorical=True, dynamic_scale=False)

        fig = make_subplots(rows=1, cols=2, subplot_titles=("Confusion Matrix (CV)", "CV ROC"))
        fig.add_trace(go.Heatmap(z=cmatrix_cv, x=['P:0', 'P:1'], y=['T:0', 'T:1'], colorscale='Blues', text=[[str(v) for v in r] for r in cmatrix_cv], texttemplate="%{text}", showscale=False), row=1, col=1)
        fig.update_yaxes(autorange="reversed", row=1, col=1)

        step = max(1, len(m['fpr']) // 100)
        fpr_plot = m['fpr'][::step]
        tpr_plot = m['tpr'][::step]
        fig.add_trace(go.Scatter(x=fpr_plot, y=tpr_plot, fill='tozeroy'), row=1, col=2)
        fig.update_layout(height=300, margin=dict(t=30, b=10, l=10, r=10), showlegend=False, template="plotly_white")

        out_plot = widgets.Output()
        with out_plot: display(go.FigureWidget(fig))

        valid_geojson_path = os.path.join(RESULTS_DIR, f"validation_full_{train_hash}.geojson")
        if not os.path.exists(valid_geojson_path):
            save_results_to_geojson(df_full_results, valid_geojson_path, role='TRAIN', aggregate=False)
        else:
            with out_log: print(f"    Skipping: {os.path.basename(valid_geojson_path)} already exists.")

        valid_conf_geojson_path = os.path.join(RESULTS_DIR, f"validation_confusion_{train_hash}.geojson")
        if not os.path.exists(valid_conf_geojson_path):
            save_confusion_geojson(df_full_results, valid_conf_geojson_path, role='TRAIN')
        else:
            with out_log: print(f"    Skipping: {os.path.basename(valid_conf_geojson_path)} already exists.")

        header = widgets.HBox([
            widgets.HTML(f"<div style='color: black; font-weight:bold;'>ACC: {m['acc']:.3f} | AUC: {m['auc']:.3f}</div>"),
            widgets.HBox([
                create_save_to_drive_button(valid_geojson_path, f"validation_full_{train_hash}", title="Save Full GeoJSON"),
                create_save_to_drive_button(valid_conf_geojson_path, f"validation_confusion_{train_hash}", title="Save Confusion GeoJSON")
            ])
        ], layout=widgets.Layout(justify_content='space-between', align_items='center', margin='0 0 10px 0', border_bottom='1px solid #ccc', padding='5px 0'))

        show_floating_panel([header, out_plot], "Validation (CV)", 'valid', width='650px')

        with out_log: print(">> Validation completed (out-of-fold predictions).")

    except Exception as e:
        print(f"ERROR: {e}")
    finally:
        set_ui_state(False)


# -----------------------------------------------------------------------------
# PREDICTION (uses updated predict_spacetime from CELL 5)
# -----------------------------------------------------------------------------
def on_pred_click(b):
    """
    Handler for "Run Prediction" button.
    Uses the updated predict_spacetime function that applies .unmask(0)
    and a robust reducer for rainfall extraction.
    """
    set_ui_state(True)
    keep_colab_alive()
    ensure_drive_mounted(out_log)
    out_log.clear_output(wait=True)

    base_date = w_pred_date.value
    with out_log:
        try:
            # --- 1. Preliminary checks ---
            if APP['model'] is None:
                print("ERROR: Run Calibration first!")
                set_ui_state(False)
                return

            if APP.get('prediction_df') is None or APP.get('prediction_df').empty:
                print("ERROR: Static morphology missing. Run 'Calc Morphometry' first.")
                set_ui_state(False)
                return

            train_hash = APP.get('current_hash', 'default')
            target_date_str = base_date.strftime('%Y-%m-%d')
            best_days = APP.get('best_days', 7)
            print(f"Model based on {best_days}-day rainfall.")
            print(f"\n--- PREDICTION DATE: {target_date_str} ---")

            # --- Suffix for categorical runs ---
            cat_suffix = "_with_cat" if CATEGORICAL_METRICS else ""

            # --- 2. Cache paths ---
            pred_csv_path = os.path.join(RESULTS_DIR, f"cache_prediction_{train_hash}_{target_date_str}{cat_suffix}.csv")
            pred_prog = pred_csv_path + ".progress"

            # --- 3. Load static prediction area data ---
            df_base = APP['prediction_df'].copy()

            # Ensure all predictor columns exist
            for col in APP.get('original_predictors', []):
                if col not in df_base.columns:
                    df_base[col] = 0.0

            # --- 4. Resume from checkpoint if available ---
            start_chunk = 0
            if os.path.exists(pred_csv_path) and os.path.exists(pred_prog):
                with open(pred_prog, 'r') as f:
                    start_chunk = int(f.read().strip())
                print(f"    Resuming predictions from row {start_chunk}...")
            elif os.path.exists(pred_csv_path):
                print(f"    Full prediction CSV already exists for {target_date_str}{cat_suffix}.")
                start_chunk = len(df_base)

            # --- 5. Run the prediction (static + dynamic) ---
            if start_chunk < len(df_base):
                print("    Running spatio-temporal prediction...")

                result_df = predict_spacetime(
                    target_date_str=target_date_str,
                    static_polygons_df=df_base,
                    model=APP['model'],
                    original_predictors=APP['original_predictors'],
                    dummies_map=APP['dummies_map'],
                    best_days=best_days,
                    train_ref_rain=None,
                    log_widget=out_log
                )

                # Save full result to CSV (checkpoint)
                result_df.to_csv(pred_csv_path, index=False)
                if os.path.exists(pred_prog):
                    os.remove(pred_prog)

                print(f"    Prediction completed for {len(result_df)} polygons.")
            else:
                # Load already computed results
                result_df = pd.read_csv(pred_csv_path)

            # --- 6. Visualise on map (with categorical suffix if used) ---
            layer_name = f'Prediction Map {target_date_str}{cat_suffix}'
            add_map_layer_geemap_native(
                result_df,
                'Final_Dynamic_Susceptibility',
                layer_name,
                VIS_PALETTE,
                dynamic_scale=False
            )

            with out_log:
                print(f"    Susceptibility: min={result_df['Final_Dynamic_Susceptibility'].min():.4f}, "
                      f"max={result_df['Final_Dynamic_Susceptibility'].max():.4f}")
                print(f"    Polygons with rain>0: {(result_df['Rn_m'] > 0).sum()}")

            # --- 7. Histogram of dynamic susceptibility ---
            last_probs = result_df['Final_Dynamic_Susceptibility'].values
            counts, bins = np.histogram(last_probs, bins=50)
            fig = go.Figure(data=[go.Bar(x=bins[:-1], y=counts, marker_color='red')])
            fig.update_layout(
                title=f"Risk Distribution ({target_date_str})",
                height=300,
                margin=dict(t=30, b=10, l=10, r=10),
                template="plotly_white"
            )

            out_plot = widgets.Output()
            with out_plot:
                display(go.FigureWidget(fig))

            # --- 8. Save buttons (GeoJSON) with suffix if needed ---
            pred_geojson_path = os.path.join(RESULTS_DIR, f"prediction_full_{train_hash}_{target_date_str}{cat_suffix}.geojson")
            if not os.path.exists(pred_geojson_path):
                save_results_to_geojson(result_df, pred_geojson_path, role='PRED', aggregate=False)
            else:
                with out_log:
                    print(f"    Skipping: {os.path.basename(pred_geojson_path)} already exists.")

            # Floating panel with results (main panel remains visible)
            header = widgets.HBox([
                widgets.HTML(f"<div style='color: black; font-weight:bold;'>Max Susceptibility: {last_probs.max():.4f}</div>"),
                create_save_to_drive_button(
                    pred_geojson_path,
                    f"prediction_full_{train_hash}_{target_date_str}{cat_suffix}",
                    title="Save GeoJSON to Drive"
                )
            ], layout=widgets.Layout(
                justify_content='space-between',
                align_items='center',
                margin='0 0 10px 0',
                border_bottom='1px solid #ccc',
                padding='5px 0'
            ))

            show_floating_panel([header, out_plot], "Prediction Results", 'pred', width='450px')

            print(f">> PREDICTION COMPLETED FOR {target_date_str}.")

        except Exception as e:
            print(f"Error during prediction: {e}")
            import traceback
            print(traceback.format_exc())
        finally:
            set_ui_state(False)


# -----------------------------------------------------------------------------
# GEOJSON EXPORT HELPERS
# -----------------------------------------------------------------------------
def save_results_to_geojson(df_results, file_path, role='TRAIN', aggregate=False, agg_col=None, agg_func='mean'):
    """Export results to GeoJSON by matching poly_uid to stored geometries."""
    target_names = [name for name, r in APP.get('poly_roles', {}).items() if r == role]
    if not target_names:
        print(f"   No polygons found for role {role}")
        return

    uid_to_geom = {}
    for name in target_names:
        poly_geoms = RAW_DATA['poly_dict'][name]['geoms']
        poly_uids = RAW_DATA['poly_dict'][name].get('poly_uids', [])
        for idx, geom in enumerate(poly_geoms):
            uid = poly_uids[idx] if idx < len(poly_uids) else None
            if uid is not None:
                uid_to_geom[uid] = geom

    if not uid_to_geom:
        print(f"   No geometries found for role {role}")
        return

    df_filtered = df_results[df_results['poly_uid'].isin(uid_to_geom.keys())].copy()
    if df_filtered.empty:
        print(f"   No matching data for role {role}")
        return

    if aggregate:
        if agg_col is None:
            raise ValueError("agg_col must be specified when aggregate=True")
        if agg_func == 'mean':
            agg_df = df_filtered.groupby('poly_uid')[agg_col].mean().reset_index()
        elif agg_func == 'max':
            agg_df = df_filtered.groupby('poly_uid')[agg_col].max().reset_index()
        elif agg_func == 'min':
            agg_df = df_filtered.groupby('poly_uid')[agg_col].min().reset_index()
        elif agg_func == 'first':
            agg_df = df_filtered.groupby('poly_uid')[agg_col].first().reset_index()
        else:
            raise ValueError(f"Unknown agg_func: {agg_func}")
        other_cols = [c for c in df_filtered.columns if c not in ['poly_uid', agg_col]]
        first_rows = df_filtered.groupby('poly_uid')[other_cols].first().reset_index()
        result_df = pd.merge(agg_df, first_rows, on='poly_uid', how='left')
    else:
        result_df = df_filtered.copy()

    result_df['geometry'] = result_df['poly_uid'].map(uid_to_geom)
    result_df = result_df.dropna(subset=['geometry'])
    if result_df.empty:
        print(f"   No rows after adding geometry.")
        return

    gdf = gpd.GeoDataFrame(result_df, geometry='geometry', crs="EPSG:4326")
    gdf.to_file(file_path, driver="GeoJSON")
    print(f"   GeoJSON saved: {file_path} with {len(gdf)} records")


def save_confusion_geojson(df_results, file_path, role='TRAIN'):
    """Save only confusion class column as GeoJSON."""
    if 'conf_class' not in df_results.columns:
        print("   conf_class column not found.")
        return
    sub_df = df_results[['poly_uid', 'conf_class']].copy()
    sub_df = sub_df.drop_duplicates(subset=['poly_uid'])
    save_results_to_geojson(sub_df, file_path, role=role, aggregate=False)


# -----------------------------------------------------------------------------
# UI WIRING AND MAP SETUP
# -----------------------------------------------------------------------------
# Connect button handlers
btn_upload_start.on_click(on_upload_start)
btn_clear_cache.on_click(on_clear_cache_click)
btn_calc_morph.on_click(on_morphometry_click)
b_calib.on_click(on_calib_click)
b_valid.on_click(on_valid_click)
b_pred.on_click(on_pred_click)

# Create map
Map = geemap.Map(height='1500px', zoom_control=False, draw_control=False, fullscreen_control=True)
Map.add_control(ZoomControl(position='bottomleft'))
APP['map'] = Map

# Initial UI panel (upload & selection)
init_box = widgets.VBox([
    css_widget_1,
    widgets.HBox([btn_upload_start, btn_clear_cache]),
    w_train_selector,
    w_pred_selector,
    w_points_selector,
    btn_calc_morph,
    btn_run_analysis
], layout=widgets.Layout(padding='15px', background_color='rgba(255, 255, 255, 0.95)',
                         border_radius='10px', box_shadow='0 4px 15px rgba(0,0,0,0.1)', width='320px'))
start_ctrl = WidgetControl(widget=init_box, position='bottomright')


# Main analysis panel (calibration, validation, prediction) – always visible
btn_back_to_start = widgets.Button(
    description="Back to Setup",
    layout=widgets.Layout(width='100%', height='35px'),
    style={'button_color': '#6c757d', 'font_weight': 'bold', 'text_color': 'white'}
)

main_content = widgets.VBox([
    b_calib,
    widgets.HTML("<div style='height:5px;'></div>"),
    b_valid,
    widgets.HTML("<div style='height:5px;'></div>"),
    w_pred_date,
    widgets.HTML("<div style='height:5px;'></div>"),
    b_pred,
    widgets.HTML("<div style='height:5px;'></div>"),
    btn_back_to_start
], layout=widgets.Layout(width='100%'))

btn_toggle_main = widgets.Button(icon='minus', layout=widgets.Layout(width='30px', height='30px'), button_style='')
def toggle_main(b):
    keep_colab_alive()
    main_content.layout.display = 'none' if main_content.layout.display != 'none' else 'flex'
    btn_toggle_main.icon = 'plus' if main_content.layout.display == 'none' else 'minus'
btn_toggle_main.on_click(toggle_main)

toggle_bar = widgets.HBox([widgets.HTML(""), btn_toggle_main], layout=widgets.Layout(justify_content='flex-end', width='100%'))

main_panel = widgets.VBox([
    css_widget_2,
    toggle_bar,
    main_content
], layout=widgets.Layout(padding='10px', background_color='rgba(255,255,255,0.95)',
                         border_radius='10px', box_shadow='0 4px 15px rgba(0,0,0,0.1)', width='300px'))

# -----------------------------------------------------------------------------
# FIX: NASCONDIAMO/MOSTRIAMO I PANNELLI INVECE DI RIMUOVERLI
# -----------------------------------------------------------------------------
# Nascondiamo il pannello principale (Analysis) all'avvio
main_panel.layout.display = 'none'
main_ctrl = WidgetControl(widget=main_panel, position='bottomright')

def go_back_to_start(b):
    keep_colab_alive()

    # 1. Nascondiamo visivamente il pannello dell'analisi
    main_panel.layout.display = 'none'

    # 2. Chiudiamo e resettiamo eventuali pannelli dei risultati (calib, valid, pred)
    for key in ['calib', 'valid', 'pred']:
        if APP['panels'].get(key):
            try: APP['map'].remove_control(APP['panels'][key])
            except: pass
            APP['panels'][key] = None

    # 3. Riportiamo in vista il pannello di setup iniziale
    init_box.layout.display = 'flex'

btn_back_to_start.on_click(go_back_to_start)


def activate_analysis(b):
    """Switch from initial upload panel to main analysis panel."""
    keep_colab_alive()
    ensure_drive_mounted(out_log)
    out_log.clear_output()

    # Nascondiamo il setup e mostriamo l'analisi
    init_box.layout.display = 'none'
    main_panel.layout.display = 'flex'

    # Aggiungiamo la legenda (solo se non c'è già, per evitare duplicati)
    if APP['panels'].get('legend') is None:
        legend_ctrl = WidgetControl(widget=create_legend_widget(), position='bottomleft')
        Map.add_control(legend_ctrl)
        APP['panels']['legend'] = legend_ctrl

btn_run_analysis.on_click(activate_analysis)

# Aggiungiamo SUBITO entrambi i controlli alla mappa per non perdere il comm_id
Map.add_control(start_ctrl)
Map.add_control(main_ctrl)

# Display map and logs
display(Map)
display(widgets.VBox([
    widgets.HTML('<div style="color: WHITE; font-weight:bold; font-size:14px; margin-bottom:5px;">ACTIVITY LOG:</div>'),
    download_area,
    out_log
], layout=widgets.Layout(margin='20px 0 0 0')))
display(Javascript('''setTimeout(() => {google.colab.output.setIframeHeight(2500, true, {maxHeight: 5000});}, 1000);'''))